
# KPM vs. Full Eigenvalue Normal Mode Analysis for PDMS Networks

Normal Mode Analysis (NMA) in :class:`~pylimer_tools_cpp.NormalModeAnalyzer`
predicts the linear viscoelastic storage modulus $G'(\omega)$ and loss
modulus $G''(\omega)$ of entangled polymer networks from their bead-spring
connectivity graph (:cite:t:`gusev_molecular_2024`). Computing the full
eigenvalue spectrum scales as $\mathcal{O}(N_{\text{atoms}}^3)$, which
quickly becomes prohibitive for realistic representative volume elements (RVEs).
The **Kernel Polynomial Method (KPM)** approximates the spectral density using
Chebyshev moments and stochastic trace estimation in $\mathcal{O}(N_{\text{atoms}})$
time.

Here we compare KPM (with $M = 1024$ Chebyshev moments and $20$
Rademacher random vectors) against full eigenvalue decomposition for realistic
Gaussian bead-spring models of polydimethylsiloxane (PDMS,
$\rho_{\text{bead}} \approx 2.23\,\text{nm}^{-3}$,
$\rho_{\text{ent}} \approx 0.0608\,\text{nm}^{-3}$,
$\langle b^2 \rangle \approx 1.11\,\text{nm}^2$) generated via
:class:`~pylimer_tools_cpp.MCUniverseGenerator` with on-demand entanglement
sampling (:meth:`~pylimer_tools_cpp.MCUniverseGenerator.sample_entanglements`).
We examine:

1. **Chain length** $N$,
2. **Cross-link conversion** $p$,
3. **Stoichiometric imbalance** $r = f N_{\text{xlinks}} / (2 N_{\text{strands}})$, and
4. **Representative volume, sample count, and computational scaling**.


In [ ]:
from pathlib import Path
import sys
import time

import numpy as np

if "__file__" in globals():
    _examples_dir = str(Path(__file__).resolve().parents[1])
    _nma_dir = str(Path(__file__).resolve().parent)
    for _p in (_examples_dir, _nma_dir):
        if _p not in sys.path:
            sys.path.insert(0, _p)

from _example_cache import (  # noqa: E402
    format_benchmark_footnote,
    get_example_cache_path,
    load_or_compute_cache,
)
from _nma_plotting import (  # noqa: E402
    plot_kpm_vs_exact_chain_length,
    plot_kpm_vs_exact_conversion,
    plot_kpm_vs_exact_rve_and_scaling,
    plot_kpm_vs_exact_stoichiometry,
)
from _nma_study_runners import run_kpm_vs_exact_pdms_study  # noqa: E402
from pylimer_tools.io.bead_spring_parameter_provider import (  # noqa: E402
    ParameterType,
    Parameters,
    get_parameters_for_polymer,
)
import pylimer_tools_cpp as pt  # noqa: E402

## Single PDMS Realization with On-Demand Entanglement Sampling
Using :func:`~pylimer_tools.io.bead_spring_parameter_provider.get_parameters_for_polymer`
with ``ParameterType.GAUSSIAN``, we generate a $L = 12\,\text{nm}$ PDMS
network, sample entanglements on demand, add the entanglement springs as bonds
in the generated :class:`~pylimer_tools_cpp.Universe`, and compare exact vs.
KPM moduli.



In [ ]:
params = get_parameters_for_polymer("PDMS", parameter_type=ParameterType.GAUSSIAN)
assert isinstance(params, Parameters)
dist_unit = params.get("distance_units")
b = params.get("<b>").to(dist_unit).magnitude
b2 = params.get("<b^2>").to(dist_unit**2).magnitude
gamma_conv = params.get_gamma_conversion_factor().to("MPa").magnitude

box_l = 12.0
vol = box_l**3
strand_len, p_demo, r_demo = 100, 0.95, 1.0
n_strands = int(round(params.get_bead_density() * vol / (strand_len + 0.5 * r_demo)))
n_xlinks = int(round(0.5 * r_demo * n_strands))

gen = pt.MCUniverseGenerator(box_l, box_l, box_l)
gen.set_seed(42)
gen.config_nr_of_mc_steps(0)
gen.set_bead_distance(b)
gen.set_mean_squared_bead_distance(b2)
gen.add_strands(n_strands, [strand_len] * n_strands)
gen.add_crosslinkers(n_xlinks, 4)
gen.link_strands_to_conversion(p_demo)

es = gen.sample_entanglements(
    nr_of_entanglements_to_sample=int(params.get_entanglement_density() * vol),
    upper_cutoff=params.get_sampling_cutoff(),
    seed="demo_nma",
    filtering_mode=pt.EntanglementFilteringMode.OMIT_INACTIVE,
)
atom_pairs = es.to_atom_pair_entanglements(gen.get_crosslinker_universe())

universe = gen.get_universe()
pairs = atom_pairs.pairs_of_atoms
if pairs:
    universe.add_bonds([pair[0] for pair in pairs], [pair[1] for pair in pairs])

edges = universe.get_edges()
nma = pt.NormalModeAnalyzer(
    edges["edge_from"], edges["edge_to"], universe.get_nr_of_atoms()
)

freqs_demo = np.logspace(-3, 2, 50)
t0 = time.perf_counter()
nma.find_all_eigenvalues(
    solver=pt.NormalModeSolver.AUTO,
    compute_eigenvectors=False,
    decompose_components=True,
)
gp_exact_demo = nma.evaluate_storage_modulus(freqs_demo) * gamma_conv / vol
t_exact_demo = time.perf_counter() - t0

demo_methods = [
    (
        "KPM",
        pt.StochasticSpectralMethod.KPM,
        {"num_steps": 1024, "num_random_vectors": 20},
    ),
    (
        "SLQ",
        pt.StochasticSpectralMethod.SLQ,
        {"num_steps": 200, "num_random_vectors": 20},
    ),
    (
        "Hybrid",
        pt.StochasticSpectralMethod.HYBRID,
        {"num_steps": 200, "num_random_vectors": 20, "num_exact_low_modes": 100},
    ),
]
print(
    f"Demo PDMS box (L={box_l:.0f} nm, N_atoms={universe.get_nr_of_atoms()}, "
    f"N_ent={len(pairs)}):"
)
print(
    f"{'Method':<8} | {'Time [s]':>8} | {'Speedup':>8} | {"Mean Rel. Err. G' [%]":>21} | {"Max Rel. Err. G' [%]":>20}"
)
print("-" * 77)
print(f"{'Exact':<8} | {t_exact_demo:8.3f} | {'1.0x':>8} | {'0.00':>21} | {'0.00':>20}")
for name, method, kw in demo_methods:
    t0 = time.perf_counter()
    gp_approx_demo = (
        nma.evaluate_storage_modulus_approx(freqs_demo, method=method, **kw)
        * gamma_conv
        / vol
    )
    dt_demo = time.perf_counter() - t0
    rel_err = (
        np.abs(gp_approx_demo - gp_exact_demo)
        / np.maximum(gp_exact_demo, 1e-12)
        * 100.0
    )
    print(
        f"{name:<8} | {dt_demo:8.3f} | {t_exact_demo / dt_demo:7.1f}x | "
        f"{np.mean(rel_err):21.2f} | {np.max(rel_err):20.2f}"
    )

cache_path = get_example_cache_path(
    "kpm_vs_exact_pdms_cache.npz", "normal_mode_analysis"
)
data = load_or_compute_cache(cache_path, run_kpm_vs_exact_pdms_study)

## Effect of Chain Length ($N$)
Varying the precursor strand length $N \in \{50, 100, 200, 400\}$ at
fixed $p = 0.95$ and $r = 1.0$ ($L = 14\,\text{nm}$) shows
that shorter strands ($N = 50$) shift the relaxation transition toward
higher frequencies and raise the low-frequency storage modulus $G'(\omega)$
due to the higher cross-link density. KPM reproduces both $G'(\omega)$
and $G''(\omega)$ accurately across the transition and Rouse regimes
($\omega\tau_0 \gtrsim 10^{-2}$), with minor stochastic/resolution
deviations only in the lowest-frequency tail where very few eigenvalues
reside.



In [ ]:
plot_kpm_vs_exact_chain_length(data)

## Effect of Cross-link Conversion ($p$)
Varying the cross-link conversion $p \in \{0.75, 0.85, 0.925, 0.98\}$ at
$N = 100$ and $r = 1.0$ approaches the classical gelation threshold
$p_{\text{gel}} = 1 / \sqrt{r(f - 1)} \approx 0.577$ from above. Lower
conversion reduces the elastically active network fraction and increases the
abundance of dangling and soluble modes, softening $G'(\omega)$ at low
frequencies while leaving the high-frequency segmental Rouse response
($\omega\tau_0 \gtrsim 0.1$) largely unchanged. KPM captures this
conversion-dependent softening faithfully without computing any eigenvalues.



In [ ]:
plot_kpm_vs_exact_conversion(data)

## Effect of Stoichiometric Imbalance ($r$)
Varying the stoichiometric ratio $r = f N_{\text{xlinks}} / (2 N_{\text{strands}})$
across $r \in \{0.65, 0.80, 1.00, 1.25\}$ (each reacted to 95% of its
stoichiometric maximum conversion, $p = 0.95 \min(1, 1/r)$) highlights
the asymmetry between cross-linker deficiency ($r < 1$) and excess
($r > 1$). Under cross-linker deficiency ($r = 0.65$), fewer
junctions connect the strands, lowering $G'(\omega)$ in the low-frequency
regime, whereas excess cross-linker ($r = 1.25$) maintains a higher
strand conversion and modulus closer to the stoichiometric reference
($r = 1.00$). Across all $r$, KPM tracks the exact spectra closely.



In [ ]:
plot_kpm_vs_exact_stoichiometry(data)

## Representative Volume, Sample Count, and Computational Scaling
Evaluating the reference PDMS network ($N = 100$, $p = 0.95$,
$r = 1.0$) across cubic box sizes $L \in [10, 36]\,\text{nm}$
(from $\sim 2.2\times 10^3$ up to $\sim 1.04\times 10^5$ beads,
$n_{\text{samples}} = 12$ realizations per box size) illustrates why KPM
is essential for realistic RVE sizes. Full eigenvalue decomposition scales as
$\mathcal{O}(N_{\text{atoms}}^3)$, becoming hundreds to thousands of
times slower than KPM by $L = 20\,\text{nm}$ ($\sim 17{,}800$
beads), whereas KPM scales linearly as $\mathcal{O}(N_{\text{atoms}})$
and evaluates $L = 36\,\text{nm}$ ($\sim 1.04\times 10^5$ beads)
in $\sim 9.8\,\text{s}$. Larger boxes also suppress finite-size scatter,
reducing the number of realizations $n_{\text{samples}}$ required for a
target 95% confidence interval.



In [ ]:
print(
    "\nAccuracy comparison across parameter sweeps (mean / max rel. error vs. Exact [%]):"
)
print(
    f"{'Method':<8} | {'Study':<13} | {"Mean Err G' [%]":>15} | {"Max Err G' [%]":>14} | {"Mean Err G'' [%]":>16} | {"Max Err G'' [%]":>15}"
)
print("-" * 94)
for method_key, method_label in (("kpm", "KPM"), ("slq", "SLQ"), ("hybrid", "Hybrid")):
    for prefix, study_name in (
        ("n", "Chain Length"),
        ("p", "Conversion"),
        ("r", "Stoichiometry"),
    ):
        gp_ex = data[f"{prefix}_gp_ex_mean"]
        gpp_ex = data[f"{prefix}_gpp_ex_mean"]
        gp_ap = data[f"{prefix}_gp_{method_key}_mean"]
        gpp_ap = data[f"{prefix}_gpp_{method_key}_mean"]
        err_gp = np.abs(gp_ap - gp_ex) / np.maximum(gp_ex, 1e-12) * 100.0
        err_gpp = np.abs(gpp_ap - gpp_ex) / np.maximum(gpp_ex, 1e-12) * 100.0
        print(
            f"{method_label:<8} | {study_name:<13} | {np.mean(err_gp):15.2f} | "
            f"{np.max(err_gp):14.2f} | {np.mean(err_gpp):16.2f} | {np.max(err_gpp):15.2f}"
        )

print("\nRuntime comparison per network (single-threaded worker, mean over repeats):")
print(
    f"{'L [nm]':>7} | {'N_atoms':>8} | {'N_ent':>6} | {'Exact [s]':>10} | "
    f"{'KPM [s]':>9} | {'SLQ [s]':>9} | {'Hybrid [s]':>10}"
)
print("-" * 73)
for i, box_l_val in enumerate(data["rve_box_lengths"]):
    n_at = float(np.mean(data["rve_n_atoms"][i]))
    n_ent_val = float(np.mean(data["rve_n_ent"][i]))
    t_kpm_val = float(np.mean(data["rve_t_kpm"][i]))
    t_slq_val = float(np.mean(data["rve_t_slq"][i]))
    t_hyb_val = float(np.mean(data["rve_t_hybrid"][i]))
    t_ex_arr = data["rve_t_ex"][i]
    if np.any(np.isfinite(t_ex_arr)):
        t_ex_val = float(np.nanmean(t_ex_arr))
        t_ex_str = f"{t_ex_val:9.2f}s"
    else:
        t_ex_str = "   skipped"
    print(
        f"{box_l_val:7.0f} | {n_at:8.0f} | {n_ent_val:6.0f} | {t_ex_str} | "
        f"{t_kpm_val:8.3f}s | {t_slq_val:8.3f}s | {t_hyb_val:9.3f}s"
    )

plot_kpm_vs_exact_rve_and_scaling(data)

## Benchmark Hardware
<div class="alert alert-info"><h4>Note</h4>

The cached runtime scaling benchmarks above were recorded on the
hardware reported below.

</div>



In [ ]:
print(format_benchmark_footnote(data))